In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch

# Create tensors
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.randn(3)
print("x:", x)
print("y:", y)

In [ ]:
# 2. Create TensorDataset objects

print("Shape of x:", x.shape)

In [ ]:
# 3. Create DataLoaders

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# Element-wise operations
print("Addition:", a + b)
print("Multiplication:", a * b)

# Matrix multiplication
A = torch.randn(2, 3)
B = torch.randn(3, 2)
C = A @ B


In [ ]:
# 4. Print shape of one batch

# 1️⃣ Flatten - Convert any shape to (batch_size, features)
x = torch.randn(32, 3, 28, 28)
x_flat = x.flatten(start_dim=1)
print("Flatten:", x_flat.shape)  # (32, 2352)

# 2️⃣ Squeeze - Remove dimensions with size 1
x = torch.randn(1, 3, 28, 28)
x_sq = x.squeeze()
print("Squeeze:", x_sq.shape)  # (3, 28, 28)

# 3️⃣ Unsqueeze - Add a new dimension of size 1
x = torch.randn(3, 28, 28)
x_unsq = x.unsqueeze(0)
print("Unsqueeze:", x_unsq.shape)  # (1, 3, 28, 28)

# 4️⃣ View - Reshape freely while keeping same number of elements
x = torch.randn(32, 28, 28, 3)
x_view = x.view(32, -1)  # Flatten all except batch
print("View:", x_view.shape)  # (32, 28*28*3)

In [ ]:
# 5. Display sample images

# Create a float32 tensor
x = torch.tensor([1.2, 2.3, 3.4], dtype=torch.float32)
print(x.dtype)  # Output: torch.float32

# Convert to float16
x_half = x.to(torch.float16)
print(x_half.dtype)  # Output: torch.float16

In [ ]:
# Task 1: Write your model class here:
class NN3Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim):
    super(NN3Layer, self).__init__()

    # First linear layer: input features -> hidden layer
    self.layer1 = nn.Linear(input_dim, hidden_dim)

    # Second linear layer: hidden layer -> hidden layer
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)

    # Output layer: hidden layer -> single continuous value
    self.layer3 = nn.Linear(hidden_dim, 1)

    # ReLU activation for non-linearity in hidden layers
    self.relu = nn.ReLU()

  # Defines how input data flows through the network
  def forward(self, x):
    # First hidden layer
    a1 = self.relu(self.layer1(x))

    # Second hidden layer
    a2 = self.relu(self.layer2(a1))

    # Output layer (regression output)
    output = self.layer3(a2)

    return output

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # Set the model to training mode
  model.train()

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # Move batch to the selected device
    X_batch = X_batch.to(device)              # shape: (batch_size, num_features)
    y_batch = y_batch.view(-1, 1).to(device) # shape: (batch_size, 1)

    # Forward pass (continuous output)
    outputs = model(X_batch)                  # shape: (batch_size, 1)
    loss = criterion(outputs, y_batch)

    # Backward pass & optimization
    optimizer.zero_grad()   # Clear previous gradients
    loss.backward()         # Compute gradients
    optimizer.step()        # Update model parameters

    running_loss += loss.item()

  # Average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
  # Set the model to evaluation mode
  model.eval()

  running_loss = 0.0

  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      # Move data to device
      X_batch = X_batch.to(device)               # shape: (batch_size, num_features)
      y_batch = y_batch.view(-1, 1).to(device)  # shape: (batch_size, 1)

      # Forward pass (continuous output)
      outputs = model(X_batch)                   # shape: (batch_size, 1)
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  # Average loss over all batches
  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
# TODO: Set up the device (use GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Task 5: start rtaining for 20 epechs:
# Model parameters
input_dim = X_train.shape[1]   # Number of tabular features
hidden_dim = 64                # Design choice

# Instantiate regression model
model = NN3Layer(input_dim, hidden_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2 (Bonus): Write your code here: